In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

In [0]:
df_dim_movies = (
    spark.table("workspace.silver.tb_info_filmes")
    # pdf pede a chave natural como string (na silver ainda é numérica, veio do csv)
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .withColumn("sk_movie_id", F.monotonically_increasing_id().cast("bigint"))
    .select(
        "sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
        "duracao_minutos", "idioma_original", "status_filme", "sinopse",
    )
)

df_dim_movies.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_movies")
df_dim_movies = spark.table("workspace.gold.dim_movies")  # relê a versão materializada, sk_movie_id fica estável pros próximos joins
display(df_dim_movies)

In [0]:
# catálogo único de gêneros, já vem limpo da silver (só falta desduplicar e criar a sk).
df_dim_genres = (
    spark.table("workspace.silver.tb_generos")
    .select("genero").distinct()
    .withColumnRenamed("genero", "nome_genero")
    .withColumn("sk_genre_id", F.monotonically_increasing_id().cast("bigint")) #criada
    .select("sk_genre_id", "nome_genero")
)

df_dim_genres.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_genres")
df_dim_genres = spark.table("workspace.gold.dim_genres")
display(df_dim_genres)

In [0]:
# só pessoas vem pra ca (produtora vira dim_companies à parte)a mesma pessoa em papéis diferentes vira duas linhas aqui, porque tipo_pessoa é parte da identidade da dimensão.

NOMES_INVALIDOS = {
    "none", "nenhum", "n/a", "na", "unknown", "não informado",
    "null", "nan", "undefined", "[]", "{}", "-", "--",
}  # placeholders de "sem valor" que vazaram como se fossem nome de pessoas, descoberto ao testar retornos

df_dim_people = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .filter(~F.lower(F.trim(F.col("nome_pessoa"))).isin(list(NOMES_INVALIDOS)))
    .distinct()
    .withColumn("sk_person_id", F.monotonically_increasing_id().cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

df_dim_people.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_people")
df_dim_people = spark.table("workspace.gold.dim_people")
display(df_dim_people)

In [0]:
df_dim_companies = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .filter(~F.lower(F.trim(F.col("nome_produtora"))).isin(list(NOMES_INVALIDOS)))
    .distinct()
    .withColumn("sk_company_id", F.monotonically_increasing_id().cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)

df_dim_companies.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_companies")
df_dim_companies = spark.table("workspace.gold.dim_companies")
display(df_dim_companies)

In [0]:
# resume as avaliações individuais em uma métrica por filme (contagem + nota média).
df_dim_reviews = (
    spark.table("workspace.silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios"),
    )
    # mesmo tipo de id_filme usado na movies, pra o join não falhar silenciosamente por tipo diferente
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), "id_filme")
    .withColumn("sk_review_id", F.monotonically_increasing_id().cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

df_dim_reviews.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_reviews")
display(df_dim_reviews)

In [0]:
df_bridge_movie_genre = (
    spark.table("workspace.silver.tb_generos")
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), "id_filme")
    .join(
        df_dim_genres.select("sk_genre_id", F.col("nome_genero").alias("genero")),
        "genero",
    )
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates()  # resolve o n:n filme<->gênero sem duplicar
)

df_bridge_movie_genre.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.bridge_movie_genre")
display(df_bridge_movie_genre)

In [0]:
df_bridge_movie_person = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), "id_filme")
    .join(
        df_dim_people.select(
            "sk_person_id",
            F.col("nome_pessoa").alias("nome_entidade"),
            F.col("tipo_pessoa").alias("tipo_entidade"),
        ),
        ["nome_entidade", "tipo_entidade"],  # essa dupla identifica uma pessoa única no de pessoas
    )
    .select("sk_movie_id", "sk_person_id")
    .dropDuplicates()
)

df_bridge_movie_person.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.bridge_movie_person")
display(df_bridge_movie_person)

In [0]:
df_bridge_movie_company = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), "id_filme")
    .join(
        df_dim_companies.select("sk_company_id", F.col("nome_produtora").alias("nome_entidade")),
        "nome_entidade",
    )
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates()
)

df_bridge_movie_company.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.bridge_movie_company")
display(df_bridge_movie_company)

In [0]:
def unico_por_filme(df):
    # row_number defensivo: financeiro/metricas não passaram por dedupe explícito na silver
    # (diferente da tb_info_filmes), então isso garante 1 linha por id_filme antes do join
    janela = Window.partitionBy("id_filme").orderBy(F.lit(1))
    return (
        df.withColumn("_rn", F.row_number().over(janela))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

df_financeiro = unico_por_filme(
    spark.table("workspace.silver.tb_financeiro_filmes").withColumn("id_filme", F.col("id_filme").cast("string"))
)
df_metricas = unico_por_filme(
    spark.table("workspace.silver.tb_metricas_engajamento").withColumn("id_filme", F.col("id_filme").cast("string"))
)

df_fact_movies_performance = (
    df_dim_movies
    .filter(F.col("status_filme") == "Lançado")  # grão = só filme lançado
    .select("sk_movie_id", "id_filme", "ano_lancamento")
    .join(df_financeiro, "id_filme", "left")
    .join(df_metricas, "id_filme", "left")
    .withColumn(
          "popularidade",
          F.when(
              (F.col("popularidade") == F.floor(F.col("popularidade")))  # sem casa decimal (popularidade real do tmdb quase sempre tem)
              & F.col("popularidade").between(1900, F.year(F.current_date()) + 5),  # e cai numa faixa que parece ano, sinal de column shift
              F.lit(None),
          ).otherwise(F.col("popularidade")),
      )
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb"),
    )
)

df_fact_movies_performance.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_movies_performance")
display(df_fact_movies_performance)

#### GenAI

In [0]:
df_atores_por_filme = (
    spark.table("workspace.gold.bridge_movie_person")
    .join(spark.table("workspace.gold.dim_people"), "sk_person_id")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_set("nome_pessoa")).alias("atores_principais")) 
    # collect_set pq não existe ordem/billing na origem, então lista todo o elenco vinculado ao filme
)

display(df_atores_por_filme)

In [0]:
df_diretores_por_filme = (
    spark.table("workspace.gold.bridge_movie_person")
    .join(spark.table("workspace.gold.dim_people"), "sk_person_id")
    .filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_set("nome_pessoa")).alias("diretor"))  # collect_set cobre os casos raros de co-direção
)

display(df_diretores_por_filme)

In [0]:
df_contexto = (
    spark.table("workspace.gold.dim_movies")
    .join(
        spark.table("workspace.gold.fact_movies_performance").select("sk_movie_id", "receita_usd", "orcamento_usd"),
        "sk_movie_id", "left",  # nem todo filme tem linha de fato (só os lançados), left join pra não sumir filme da tabela
    )
    .join(df_atores_por_filme, "sk_movie_id", "left")
    .join(df_diretores_por_filme, "sk_movie_id", "left")
)

# concat() zera a string inteira se qualquer campo envolvido for nulo por isso cada pedaço do template vira um texto com fallback antes de entrar na concatenação final.
titulo_texto = F.coalesce(F.col("titulo"), F.lit("Título não informado"))
ano_texto = F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano não informado"))
receita_texto = F.coalesce(F.concat(F.lit("US$ "), F.format_number(F.col("receita_usd"), 2)), F.lit("valor de bilheteria não divulgado"))
orcamento_texto = F.coalesce(F.concat(F.lit("US$ "), F.format_number(F.col("orcamento_usd"), 2)), F.lit("orçamento não divulgado"))
atores_texto = F.coalesce(F.col("atores_principais"), F.lit("elenco não informado"))
diretor_texto = F.coalesce(F.col("diretor"), F.lit("direção não informada"))
sinopse_texto = F.coalesce(
    F.when(F.trim(F.coalesce(F.col("sinopse"), F.lit(""))) != "", F.col("sinopse")),
    F.lit("sinopse não disponível"),  # sinopse nula ou só espaço em branco também cai no fallback
)

df_gold_genai_movies_context = df_contexto.select(
    F.col("id_filme").alias("movie_id"),
    titulo_texto.alias("title"),
    F.concat(
        F.lit("O filme "), titulo_texto,
        F.lit(", lançado no ano de "), ano_texto,
        F.lit(", faturou "), receita_texto,
        F.lit(" e teve um custo de "), orcamento_texto,
        F.lit(". Estrelado por "), atores_texto,
        F.lit(" e dirigido por "), diretor_texto,
        F.lit(", o filme possui a seguinte sinopse: "), sinopse_texto,
        F.lit("."),
    ).alias("llm_context_document"),
)

df_gold_genai_movies_context.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.gold_genai_movies_context")
display(df_gold_genai_movies_context)